# 06 — Multi-Object Tracking
Kalman Filter + Hungarian algorithm tracker with persistent IDs across frames.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install ultralytics filterpy scipy opencv-python matplotlib -q

In [ ]:
import os, cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ultralytics import YOLO
from filterpy.kalman import KalmanFilter
from scipy.optimize import linear_sum_assignment

BASE_PATH  = '/content/drive/MyDrive/Project/Kitti/Training'
IMAGE_PATH = os.path.join(BASE_PATH, 'image_2/training/image_2')
LIDAR_PATH = os.path.join(BASE_PATH, 'velodyne/training/velodyne')
CALIB_PATH = os.path.join(BASE_PATH, 'calib/data_object_calib/training/calib')
RESULTS    = '/content/drive/MyDrive/Project/results'
os.makedirs(RESULTS, exist_ok=True)

In [ ]:
# Paste parse_calib, project_lidar_to_image, get_lidar_depth_for_box
# from notebook 05 here — keeping each notebook self-contained

def parse_calib(calib_path):
    calib = {}
    with open(calib_path, 'r') as f:
        for line in f.readlines():
            if ':' in line:
                key, value = line.split(':', 1)
                calib[key.strip()] = np.array(
                    [float(x) for x in value.strip().split()])
    return (calib['P2'].reshape(3,4),
            calib['R0_rect'].reshape(3,3),
            calib['Tr_velo_to_cam'].reshape(3,4))

def project_lidar_to_image(points_3d, P2, R0, Tr):
    N       = points_3d.shape[0]
    pts_hom = np.hstack([points_3d, np.ones((N,1))])
    Tr_full = np.vstack([Tr,[0,0,0,1]])
    R0_full = np.eye(4); R0_full[:3,:3] = R0
    pts_cam = R0_full @ Tr_full @ pts_hom.T
    depth   = pts_cam[2,:]; valid = depth > 0
    pts_cam = pts_cam[:,valid]; depth = depth[valid]
    pts_img = P2 @ pts_cam; pts_img = pts_img/pts_img[2,:]
    return pts_img[:2,:].T, depth, valid

def get_lidar_depth_for_box(box_2d, pixels, depth, pts_xyz, valid_mask):
    x1,y1,x2,y2 = box_2d
    px,py = pixels[:,0], pixels[:,1]
    in_box = (px>=x1)&(px<=x2)&(py>=y1)&(py<=y2)
    d = depth[in_box]
    if len(d) == 0: return None, 0
    med   = np.median(d)
    clean = d[np.abs(d-med)<2.0]
    if len(clean) == 0: return None, 0
    return round(float(clean.mean()),2), len(clean)

In [ ]:
class ObjectTracker:
    count = 0
    def __init__(self, box, class_name, distance):
        ObjectTracker.count += 1
        self.id=ObjectTracker.count; self.class_name=class_name
        self.distance=distance; self.hits=1; self.no_match=0; self.history=[]
        self.kf=KalmanFilter(dim_x=8,dim_z=4)
        self.kf.F=np.eye(8)
        for i in range(4): self.kf.F[i,i+4]=1
        self.kf.H=np.zeros((4,8)); self.kf.H[:4,:4]=np.eye(4)
        self.kf.R*=10; self.kf.P*=100; self.kf.Q*=0.1
        self.kf.x[:4]=self._to_state(box)
    def _to_state(self,box):
        x1,y1,x2,y2=box
        return np.array([[(x1+x2)/2],[(y1+y2)/2],[x2-x1],[y2-y1]],dtype=float)
    def _to_box(self):
        cx,cy,w,h=self.kf.x[:4].flatten()
        return [cx-w/2,cy-h/2,cx+w/2,cy+h/2]
    def predict(self): self.kf.predict(); return self._to_box()
    def update(self,box,distance):
        self.kf.update(self._to_state(box))
        self.distance=distance; self.hits+=1; self.no_match=0
        self.history.append(self._to_box())

class MultiObjectTracker:
    def __init__(self,iou_threshold=0.15,max_age=2):
        self.trackers=[]; self.iou_threshold=iou_threshold; self.max_age=max_age
    def _iou(self,b1,b2):
        xi1=max(b1[0],b2[0]); yi1=max(b1[1],b2[1])
        xi2=min(b1[2],b2[2]); yi2=min(b1[3],b2[3])
        inter=max(0,xi2-xi1)*max(0,yi2-yi1)
        a1=(b1[2]-b1[0])*(b1[3]-b1[1]); a2=(b2[2]-b2[0])*(b2[3]-b2[1])
        return inter/(a1+a2-inter+1e-6)
    def update(self,detections):
        predicted=[t.predict() for t in self.trackers]
        matched_d=set(); matched_t=set()
        if self.trackers and detections:
            cost=np.zeros((len(detections),len(self.trackers)))
            for d,det in enumerate(detections):
                for t,pred in enumerate(predicted):
                    cost[d,t]=1-self._iou(det['box'],pred)
            di,ti=linear_sum_assignment(cost)
            for d,t in zip(di,ti):
                if cost[d,t]<(1-self.iou_threshold):
                    self.trackers[t].update(detections[d]['box'],detections[d]['distance'])
                    matched_d.add(d); matched_t.add(t)
        for d,det in enumerate(detections):
            if d not in matched_d:
                self.trackers.append(ObjectTracker(det['box'],det['class_name'],det['distance']))
        for i,t in enumerate(self.trackers):
            if i not in matched_t: t.no_match+=1
        self.trackers=[t for t in self.trackers if t.no_match<=self.max_age]
        return self.trackers

print('Tracker defined successfully')

In [ ]:
ObjectTracker.count = 0
tracker   = MultiObjectTracker(iou_threshold=0.15, max_age=2)
model     = YOLO('yolov8n.pt')
id_colors = ['lime','red','cyan','magenta','yellow','orange','white','pink','aqua','coral']

all_ids        = sorted([f.replace('.png','') for f in os.listdir(IMAGE_PATH) if f.endswith('.png')])
consecutive_ids = all_ids[750:755]

for sid in consecutive_ids:
    img     = cv2.imread(f'{IMAGE_PATH}/{sid}.png')
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h,w     = img.shape[:2]
    pts     = np.fromfile(f'{LIDAR_PATH}/{sid}.bin',dtype=np.float32).reshape(-1,4)
    pts_xyz = pts[:,:3]
    P2,R0,Tr = parse_calib(f'{CALIB_PATH}/{sid}.txt')
    pixels,depth,valid = project_lidar_to_image(pts_xyz,P2,R0,Tr)
    px,py = pixels[:,0],pixels[:,1]
    in_bounds=(px>=0)&(px<w)&(py>=0)&(py<h)
    results   = model(f'{IMAGE_PATH}/{sid}.png',conf=0.35,classes=[0,1,2,3,5,7],verbose=False)
    boxes     = results[0].boxes.xyxy.cpu().numpy()
    scores    = results[0].boxes.conf.cpu().numpy()
    class_ids = results[0].boxes.cls.cpu().numpy()
    class_names = {0:'person',1:'bicycle',2:'car',3:'motorcycle',5:'bus',7:'truck'}
    detections = []
    for box,score,cls in zip(boxes,scores,class_ids):
        dist,n = get_lidar_depth_for_box(box,pixels,depth,pts_xyz,valid)
        detections.append({'box':box.tolist(),'class_name':class_names.get(int(cls),'unknown'),
                           'confidence':float(score),'distance':dist})
    active = tracker.update(detections)
    confirmed = [t for t in active if t.hits>=2]
    fig,ax = plt.subplots(1,figsize=(16,6))
    ax.imshow(img_rgb)
    dn=(depth-depth.min())/(depth.max()-depth.min())
    ax.scatter(px[in_bounds],py[in_bounds],c=dn[in_bounds],cmap='jet',s=1.5,alpha=0.35)
    for t in confirmed:
        box=t._to_box(); x1,y1,x2,y2=box
        color=id_colors[(t.id-1)%len(id_colors)]
        dist_str=f'{t.distance}m' if t.distance else '?m'
        rect=patches.Rectangle((x1,y1),x2-x1,y2-y1,linewidth=2.5,edgecolor=color,facecolor='none')
        ax.add_patch(rect)
        ax.text(x1,y1-6,f'ID{t.id} {t.class_name} | {dist_str} | hits={t.hits}',
                color=color,fontsize=9,fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2',facecolor='black',alpha=0.7))
    ax.set_title(f'Multi-Object Tracking — Frame {sid} ({len(confirmed)} confirmed)',fontsize=13)
    ax.axis('off'); plt.tight_layout()
    plt.savefig(f'{RESULTS}/{sid}_tracked_v2.png',dpi=150,bbox_inches='tight')
    plt.show(); plt.close()
    print(f'Frame {sid} — {len(confirmed)} confirmed tracks')
    for t in confirmed:
        print(f'  ID{t.id:02d} {t.class_name:10s} dist={t.distance}m hits={t.hits}')
    print()